In [1]:
import pandas as pd
import numpy as np
import glob
import os
import gc
import hashlib
import pyarrow
from functions import *

In [2]:
normalize_name("Città della Pieve")

'CITTA DELLA PIEVE'

Load OMI data - read - concatenate

In [3]:
# Grab all CSV files
folder = "data_origin/omi_estimate"

all_files = glob.glob(os.path.join(folder, "*.csv"))

out_dir = "datasets/omi_estimate"
os.makedirs(out_dir, exist_ok=True)


# Concatenate all datasets into a single DataFrame
dfs = []

for i,f in enumerate(all_files):
    """
    Read every dataset in the folder as input.

    Extract year and semester from the file name.

    Select relevant columns.

    Return datasets with updated istat codes. 
    """
    # Extract filename without extension
    filename = os.path.splitext(os.path.basename(f))[0]
    
    # Extract semester code
    parts = filename.split("_")
    semester_code = parts[-2]  # second to last part
    year = semester_code[:4]
    sem = semester_code[4]
    semester = f"{year}_S{sem}"
    
    # Read CSV, skip first title line
    df = pd.read_csv(f, sep=';', skiprows=1)
    
    # Strip whitespace and remove BOM from column names
    df.columns = df.columns.str.strip().str.replace('\ufeff','')

    # Semester columns: year_semester -> year + S1/S2; semester -> 1/2
    df['year_semester'] = semester
    df['semester'] = sem
    df['year'] = year

    # Keep relevant columns
    columns = [
        'Comune_ISTAT', 'Comune_descrizione', 'year', 'year_semester', 'semester', 'Zona', 
        'Descr_Tipologia', 'Stato', 'Compr_min', 'Compr_max'
        ]

    df = df[columns]
    
    # Convert numeric columns to numeric type
    numeric_cols = ['Compr_min', 'Compr_max']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '.', regex=False), errors='coerce')

    # Normalize municipality names
    df['Comune_descrizione'] = df['Comune_descrizione'].apply(normalize_name)

    # Convert buy column to int
    buy_col = ['Compr_min', 'Compr_max']
    for col in buy_col:
        df[col] = df[col].astype('Int64')

    # Extract ISTAT code from the last 6 digits of the 'Comune_ISTAT' column -
    # add missing 0s - transform to object (preserve leading 0s)
    df['Comune_ISTAT'] = df['Comune_ISTAT'].astype('Int64').astype('str').str[-6:]

    add_zeroes(df, ['Comune_ISTAT'], 6)

    df['Comune_ISTAT'] = df['Comune_ISTAT'].astype('object')

    dfs.append(df)

# Concatenate all DataFrames into one
final_df = pd.concat(dfs, ignore_index=True)

In [4]:
final_df.head()

,Comune_ISTAT,Comune_descrizione,year,year_semester,semester,Zona,Descr_Tipologia,Stato,Compr_min,Compr_max
0,006003,ALESSANDRIA,2014,2014_S1,1,B1,Abitazioni civili,NORMALE,680,960
1,006003,ALESSANDRIA,2014,2014_S1,1,B1,Box,NORMALE,1100,1600
2,006003,ALESSANDRIA,2014,2014_S1,1,B1,Posti auto coperti,NORMALE,700,1000
3,006003,ALESSANDRIA,2014,2014_S1,1,B1,Posti auto scoperti,NORMALE,600,800
4,006003,ALESSANDRIA,2014,2014_S1,1,B1,Magazzini,NORMALE,1050,1300


Translate

In [5]:
# Translate columns names
column_renames = {
    'Comune_ISTAT' : 'mun_istat', 
    'Comune_descrizione' : 'mun_name', 
    'Zona' : 'zone',
    'Descr_Tipologia' : 'type',
    'Stato' : 'condition',
    'Compr_min' : 'buy_min',
    'Compr_max' : 'buy_max'
}

final_df = final_df.rename(columns=column_renames)

# Translate values names
final_df["type"] = final_df["type"].replace({
    "Abitazioni civili": "Residential housing",
    "Box": "Garage",
    "Ville e Villini": "Independent houses and villas",
    "Negozi": "Shops",
    "Abitazioni di tipo economico": "Lowcost housing",
    "Magazzini": "Warehouses",
    "Uffici": "Offices",
    "Laboratori": "Laboratories",
    "Capannoni tipici": "Typical industrial buildings",
    "Capannoni industriali": "Industrial buildings",
    "Autorimesse": "Garages",
    "Posti auto scoperti": "Uncovered parking spaces",
    "Posti auto coperti": "Covered parking spaces",
    "Centri commerciali": "Shopping centers",
    "Uffici strutturati": "Structured offices",
    "Abitazioni tipiche dei luoghi": "Typical local housing",
    "Abitazioni signorili": "Luxury housing",
    "Pensioni e assimilati": "Guesthouses and similar",
    "Fabbricati e locali per esercizi sportivi": "Sports facilities"
})

final_df["condition"] = final_df["condition"].replace({
    "NORMALE": "Normal",
    "OTTIMO": "Excellent",
    "SCADENTE": "Poor"
})

Duplicated Istat codes

In [6]:
# Count the number of duplicate listings
duplicates = final_df.value_counts(subset=[
    'mun_istat', 'zone', 'year_semester', 'condition', 'type'
    ])

duplicates = duplicates[duplicates > 1]

print("Number of duplicate listings for the same semester:", duplicates.sum())

Number of duplicate listings for the same semester: 544


In [7]:
# Delete duplicate listings for the same semester - keep the first occurrence
final_df = final_df.drop_duplicates(subset=[
    'mun_istat', 'zone', 'year_semester', 'condition', 'type'
    ], keep='first')

Missing/0 values

In [8]:
# Check for missing values
missing_values = final_df.isnull().sum()
print("Missing values in each column:\n", missing_values)

Missing values in each column:
 mun_istat          0
mun_name           0
year               0
year_semester      0
semester           0
zone               0
type               0
condition        211
buy_min          259
buy_max          259
dtype: int64


In [9]:
# Deleting missing values
final_df = final_df.dropna()

In [10]:
# Check for 0s in buy columns
zero_values = (final_df == 0).sum()
print("Zero values in each column:\n", zero_values)

Zero values in each column:
 mun_istat            0
mun_name             0
year                 0
year_semester        0
semester             0
zone                 0
type                 0
condition            0
buy_min          19513
buy_max          19513
dtype: Int64


In [11]:
# Dropping rows with 0 'buy_min/max' values
final_df = final_df[(final_df['buy_min'] != 0) & (final_df['buy_max'] != 0)]

Update ISTAT codes

In [45]:
# ISTAT codes updated to 2025
df_new_istat = pd.read_csv('datasets/mun_istat_codes.csv')

# ISTAT codes changes
df_change = pd.read_csv('datasets/changes_istat.csv')

In [46]:
# Uniform ISTAT codes across datasets
add_zeroes(df_new_istat, ['mun_istat'], 6)
add_zeroes(df_change, ['mun_istat_old', 'mun_istat_new'], 6)

df_new_istat['mun_istat'] = df_new_istat['mun_istat'].astype('object')
df_change['mun_istat_old'] = df_change['mun_istat_old'].astype('object')
df_change['mun_istat_new'] = df_change['mun_istat_new'].astype('object')


In [77]:
# Update ISTAT codes
updated_df = update_istat(
    df=final_df,
    df_map=df_change, 
    valid_codes=df_new_istat["mun_istat"], 
    istat_col="mun_istat",
    istat_old = "mun_istat_old",
    istat_new = "mun_istat_new"
)

# Select only suppressed municipalities
supp_df = updated_df[updated_df['suppressed'].isin([True])]

supp_df = supp_df.drop(columns = ['mun_istat', 'mun_istat_updated', 'changed', 'suppressed'])

# Select only non-suppressed municipalities
updated_df = updated_df[updated_df['suppressed'].isin([False])]

updated_df = updated_df.drop(columns = ['mun_istat', 'changed', 'suppressed'])

updated_df = updated_df.rename(columns = {'mun_istat_updated' : 'mun_istat'})

df_new_istat = df_new_istat[['mun_istat','mun_name']]


In [ ]:
# Merge suppressed municipalities with new ISTAT codes (name based)
compare_df = pd.merge(supp_df, df_new_istat, on = 'mun_name', how = 'left')

# Drop municipalities with no match in new ISTAT codes
compare_df = compare_df.dropna(subset = ['mun_istat'])

compare_df['mun_istat'] = compare_df['mun_istat'].astype('str')

# Merge with updated dataset
all_df = pd.concat([updated_df, compare_df], ignore_index=True)

In [87]:
all_df[(all_df['mun_name'] == 'MONZA') & (all_df['type'] == 'Garage') & (all_df['zone'] == 'B11')]

,mun_name,year,year_semester,semester,zone,type,condition,buy_min,buy_max,mun_istat
32252,MONZA,2014,2014_S1,1,B11,Garage,Normal,2600,3900,108033
191576,MONZA,2014,2014_S2,2,B11,Garage,Normal,2700,3800,108033
351400,MONZA,2015,2015_S1,1,B11,Garage,Normal,2700,3700,108033
510977,MONZA,2015,2015_S2,2,B11,Garage,Normal,2650,3700,108033
670795,MONZA,2016,2016_S1,1,B11,Garage,Normal,2550,3600,108033
830658,MONZA,2016,2016_S2,2,B11,Garage,Normal,2550,3600,108033
988106,MONZA,2017,2017_S1,1,B11,Garage,Normal,2550,3600,108033
1146203,MONZA,2017,2017_S2,2,B11,Garage,Normal,2550,3600,108033
1305042,MONZA,2018,2018_S1,1,B11,Garage,Normal,2450,3600,108033
1463335,MONZA,2018,2018_S2,2,B11,Garage,Normal,2350,3600,108033
